# Technique 1

## Case 2: Earth to Venus


### Test settings

Convergence tests and computational time use separate cells. Each timing test uses seven serial `%%timeit` repeats with the integrators initialized beforehand.


In [1]:
import os
import sys
import platform

for name in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS',
             'VECLIB_MAXIMUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[name] = '1'

import numpy as np
import scipy
import heyoka as hy
from IPython.display import display, Markdown

hy.set_nthreads(1)



In [2]:
# =========================
# Timing Settings
# =========================
from collections import Counter

cpu_samples = []
fast_cpus = []
if sys.platform == 'darwin':
    import ctypes
    import plistlib
    import subprocess
    tree = plistlib.loads(subprocess.check_output(
        ['/usr/sbin/ioreg', '-a', '-l', '-p', 'IODeviceTree', '-n', 'cpus', '-r']))
    pending = list(tree)
    while pending:
        node = pending.pop()
        if node.get('cluster-type') == b'P\x00' and 'cpu-id' in node:
            fast_cpus.append(int.from_bytes(node['cpu-id'], 'little'))
        pending.extend(node.get('IORegistryEntryChildren', []))
    lib = ctypes.CDLL('/usr/lib/libSystem.B.dylib')
    lib.pthread_set_qos_class_self_np.argtypes = [ctypes.c_uint, ctypes.c_int]
    lib.pthread_cpu_number_np.argtypes = [ctypes.POINTER(ctypes.c_size_t)]

def set_timing_priority():
    if sys.platform == 'darwin':
        assert lib.pthread_set_qos_class_self_np(0x21, 0) == 0

def sample_cpu():
    if sys.platform == 'darwin':
        cpu = ctypes.c_size_t()
        assert lib.pthread_cpu_number_np(ctypes.byref(cpu)) == 0
        cpu_samples.append(cpu.value)

records_by_method = {}
results_by_method = {}

def save_trial(i, result, accepted, nfev):
    records.append(dict(trial=int(i), accepted=bool(accepted), nfev=int(nfev),
        status=int(result.status) if result is not None else None,
        residual_norm=float(np.linalg.norm(result.fun)) if result is not None else None,
        x=result.x.tolist() if result is not None else None))


In [3]:
import numpy as np
import heyoka as hy

# =========================
# Setup (SI Units)
# =========================
AU2m = 149597870700 # JPL 
day2s = 86400.0     # JPL
yr2day = 365.25
yr2s = yr2day*day2s

x0_nu = [ # r, v [AU, AU/yr] Time: 7 Oct. 2005 0:0:0.0
    9.708322 * 1e-1, 2.375844 * 1e-1 , -1.671055 * 1e-6,
    -1.598191, 6.081958, 9.443368 * 1e-5,
]
xf_nu = [ # r, v [AU, AU/yr] Time: 7 Oct. 2005 0:0:0.0 + 1000day
    -3.277178 * 1e-1, 6.389172 * 1e-1, 2.765929 * 1e-2,
    -6.598211, -3.412933, 3.340902 * 1e-1
]
tof = 1000.0*day2s # [s]

m0 = 1500.0 # [kg]
Tmax = 0.33 # [N]
Isp = 3800  # [s]

mu = 132712440018*1e9   # [m3∕s2]
g0 = 9.8065             # [m∕s2]

# =========================
# Nondimensionalize
# =========================
# Nondimension units
LUnit = AU2m
# TUnit = yr2day*day2s
TUnit = np.sqrt(LUnit * LUnit * LUnit / mu)
MUnit = m0

VUnit = LUnit / TUnit # VUnit to m/s
AUnit = VUnit / TUnit # AUnit to m/s^2

# x0_nu, xf_nu, t0_nu, tf_nu, m0_nu, mu_nu, c1=Tmax_nu, c2=Isp_nu*g0_nu 
t0_nu = 0.0
tf_nu = t0_nu+tof
m0_nu = m0
mu_nu = mu
Tmax_nu = Tmax
Isp_nu = Isp
g0_nu = g0

vel_scale = TUnit / yr2s
x0_nu[3:6] = [x * vel_scale for x in x0_nu[3:6]]
xf_nu[3:6] = [x * vel_scale for x in xf_nu[3:6]]

t0_nu = t0_nu / TUnit
tf_nu = tf_nu / TUnit
m0_nu = m0_nu / MUnit
mu_nu = mu_nu * TUnit * TUnit / LUnit / LUnit / LUnit
Tmax_nu = Tmax_nu / MUnit / AUnit
Isp_nu = Isp_nu / TUnit
g0_nu = g0_nu / AUnit

c1 = Tmax_nu
c2 = Isp_nu*g0_nu


In [4]:
import twobody_RV_hamilton_log

ta = twobody_RV_hamilton_log.ta
ta_var = twobody_RV_hamilton_log.ta_var
ta_var_ms = twobody_RV_hamilton_log.ta_var_ms
ta_b2 = twobody_RV_hamilton_log.ta_b2
ta_var_ms_b2 = twobody_RV_hamilton_log.ta_var_ms_b2


In [5]:
# =========================
# Define the shooting function
# =========================
def shooting(lam, eps = 1.0e-5, lambda0 = 1.0, out = False, Time_out = None):
    # lam[0:7] -> initial co-states [lx,ly,lz,lvx,lvy,lvz,lm]
    # function uses global variables ta, t0_nu, x0_nu, m0_nu, tf_nu, xf_nu, 
    # mu_nu, c1, c2
    ta.pars[:] = [mu_nu, c1, c2, eps, lambda0]
    ta.time = t0_nu
    ta.state[:7] = x0_nu + [m0_nu]
    ta.state[7:14] = lam[:]

    ta.propagate_until(tf_nu)

    # Construct 7 equations: 6 terminal Cartesian state errors + 1 terminal co-state lm = 0
    retval = [
        ta.state[0] - xf_nu[0],
        ta.state[1] - xf_nu[1],
        ta.state[2] - xf_nu[2],
        ta.state[3] - xf_nu[3],
        ta.state[4] - xf_nu[4],
        ta.state[5] - xf_nu[5],
        ta.state[13]
    ]

    if out:
        print("  Final Mass:", ta.state[6]*MUnit)

    if Time_out is not None:
        test = Time_out

    return retval

# =========================
# Define the sensitivity matrix ∂[Xf, Lf]/∂[L0, lambda0]
# =========================
def stm_14x8(lam, eps = 1.0e-5, lambda0 = 1.0, tf = tf_nu):
    """
    Return Phi(tf) with shape (14,8),
    w.r.t. [lx0, ly0, lz0, lvx0, lvy0, lvz0, lm0, lambda0].
    """
    # lam[0:7] -> initial co-states [lx,ly,lz,lvx,lvy,lvz,lm]
    # function uses global variables ta, t0_nu, x0_nu, m0_nu, tf_nu, xf_nu, 
    # mu_nu, c1, c2

    lam = np.asarray(lam, float)
    ta_var.pars[:] = [mu_nu, c1, c2, eps, lambda0]
    ta_var.time = t0_nu
    ta_var.state[:7] = list(x0_nu) + [m0_nu]
    ta_var.state[7:14] = lam.tolist()

    # Initial first-order sensitivity matrix:
    # dX0 / d[lam0, lambda0] =
    # [ 0_(7x7)  0_(7x1) ]
    # [ I_(7x7)  0_(7x1) ]
    Phi0 = np.vstack([np.zeros((7, 7)), np.eye(7)])
    Phi0 = np.hstack([Phi0, np.zeros((14, 1))])

    # heyoka stores the variational block as a flat array
    ta_var.state[14:] = Phi0.ravel().tolist()

    # Propagate
    ta_var.propagate_until(tf)

    # Extract first-order variational block
    sl = ta_var.get_vslice(order = 1)
    Phi_tf = np.asarray(ta_var.state[sl]).reshape(14, 8)

    return Phi_tf

# =========================
# Set parameters for minpack
# =========================
import time
from scipy.optimize import root
repeat_num_minpack  = 5000
# repeat_num_minpack  = 1

minpack_tol     = 1.0e-10
minpack_xtol    = 1.0e-10
minpack_maxfev  = 50000
# minpack_factor  = 1.
minpack_factor  = 0.1
minpack_diag    = [1.0]*3 + [1.0]*3 + [1.0]*1
minpack_eps     = 0.
minpack_error   = 1.0e-7

np.set_printoptions(precision=16, suppress=True)


In [6]:
# =========================
# Common random initial guess pool
# =========================
RANDOM_SEED = 322
FB_AUX_SEED = 1322

def build_common_lam0_pool(n_trials, seed=RANDOM_SEED):
    """
    Build the common 7D initial co-state pool.
    
    Draw seven components in (-0.1, 0.1), then draw lambda_m in (0, 1).
    """
    rng = np.random.RandomState(seed)
    pool = np.empty((int(n_trials), 7), dtype=float)

    for i in range(int(n_trials)):
        pool[i] = 0.1 * (2.0 * rng.random_sample(7) - 1.0)
        pool[i, 6] = 0.1 * rng.random_sample()  # lambda_m(t0) in [0, 0.1)

    return pool

def build_fb_z0_pool(lam0_pool, m0=1.0, aux_seed=FB_AUX_SEED):
    """
    Build z0 = [lam0(7), mf0(1), lam_rf0(3), lam_vf0(3)].
    
    Use lam0_pool for the initial co-states and an independent random
    stream for the terminal co-states.
    """
    lam0_pool = np.asarray(lam0_pool, dtype=float)
    if lam0_pool.ndim != 2 or lam0_pool.shape[1] != 7:
        raise ValueError("lam0_pool must have shape (n_trials, 7).")

    n_trials = lam0_pool.shape[0]
    rng_aux = np.random.RandomState(aux_seed)

    mf0 = np.full((n_trials, 1), 0.9 * float(m0), dtype=float)
    lrf0 = 0.1 * (2.0 * rng_aux.random_sample((n_trials, 3)) - 1.0)
    lvf0 = 0.1 * (2.0 * rng_aux.random_sample((n_trials, 3)) - 1.0)

    z0_pool = np.hstack([lam0_pool.copy(), mf0, lrf0, lvf0])
    assert np.array_equal(z0_pool[:, :7], lam0_pool)
    return z0_pool

lam0_pool = build_common_lam0_pool(repeat_num_minpack, seed=RANDOM_SEED)
fb_z0_pool = build_fb_z0_pool(lam0_pool, m0=m0_nu, aux_seed=FB_AUX_SEED)

print("Shared trial pool:", lam0_pool.shape)
print("Forward-backward pool:", fb_z0_pool.shape)
print("First lam0:", lam0_pool[0])


Shared trial pool: (5000, 7)
Forward-backward pool: (5000, 14)
First lam0: [-0.017485121169841  -0.0716615874106479 -0.0447910307558221
 -0.0666617749979717 -0.0965170607575046  0.0155893011895946
  0.0107421362390202]


In [7]:
# =========================
# Check the common initial guess pool
# =========================
assert lam0_pool.shape == (repeat_num_minpack, 7)
assert fb_z0_pool.shape == (repeat_num_minpack, 14)
assert np.array_equal(fb_z0_pool[:, :7], lam0_pool)

print("trial 0 lam0:", lam0_pool[0])
print("trial 1 lam0:", lam0_pool[1])


trial 0 lam0: [-0.017485121169841  -0.0716615874106479 -0.0447910307558221
 -0.0666617749979717 -0.0965170607575046  0.0155893011895946
  0.0107421362390202]
trial 1 lam0: [-0.0645120399559002 -0.0543674780240145  0.0974685476644564
 -0.0908420789513544  0.0899968244483845 -0.0646167455729844
  0.09613063823968  ]


In [8]:
def shooting_jac(lam, eps = 1.0e-5, lambda0 = 1.0):
    """
    Return the shooting Jacobian J with shape (7,7),
    w.r.t. the 7 initial co-states.
    """
    Phi_tf = stm_14x8(lam, eps, lambda0)

    J = np.zeros((7, 7))

    # Residuals 1..6: terminal Cartesian state error
    J[0:6, :] = Phi_tf[0:6, 0:7]

    # Residual 7: terminal lm(tf) = 0
    J[6, :] = Phi_tf[13, 0:7]

    return J


In [9]:
# =========================
# Integrator parameter setting (eps and lambda0 are fixed.)
# =========================
# epsf = 1.0e-5
epsf = 1.0e-3
lambda0 = 1.0
ta.pars[:]   = [mu_nu, c1, c2, epsf, lambda0]
ta_var_ms.pars[:]   = [mu_nu, c1, c2, epsf, lambda0]

# =========================
# Segment propagation and STM
# =========================
def stm_14x15_ms(Xk_14, l0, t_start, t_end, active_cols=None): # STM 14x15 with selected initial directions
    """
    Integrate one segment from t_start to t_end:
        Xk_14 = [x,y,z,vx,vy,vz,m, lx,ly,lz,lvx,lvy,lvz,lm]

    return (X_end, L_end, Phi):
        Phi = ∂[X_end; L_end]/∂[Xk; Lk; l0] ∈ R^{14×15}

    Parameters
    ----------
    active_cols : None or iterable of int
        If None, propagate the full STM with initial Phi0 = [I_14 | 0].
        Otherwise, only the specified columns are activated in Phi0.
        Inactive columns of Phi0 are initialized to zero.
    """
    ta_var_ms.time = float(t_start)

    # ============================================================
    # Initialize the selected columns of Phi0
    # Full STM initialization: [I_14 | 0_(14x1)]
    # ============================================================
    Phi0 = np.zeros((14, 15), dtype=float)

    if active_cols is None:
        # Full STM
        Phi0[:, :14] = np.eye(14)
    else:
        # Initialize the selected columns
        active_cols = list(active_cols)
        for j in active_cols:
            if not (0 <= j < 15):
                raise ValueError(f"active_cols contains invalid column index {j}, must be in [0, 14].")
            if j < 14:
                Phi0[j, j] = 1.0
            else:
                # For lambda0 column, initial sensitivity is zero by definition,
                # so keep column 14 as zero even if requested.
                pass

    ta_var_ms.state[:] = list(Xk_14) + Phi0.ravel().tolist()
    ta_var_ms.pars[4] = float(l0)

    ta_var_ms.propagate_until(float(t_end))

    sl = ta_var_ms.get_vslice(order=1)
    Phi = np.asarray(ta_var_ms.state[sl]).reshape(14, 15).copy()

    X_end = np.asarray(ta_var_ms.state[:7]).copy()
    L_end = np.asarray(ta_var_ms.state[7:14]).copy()
    return X_end, L_end, Phi


In [10]:
# =========================
# Utilities
# =========================
def _ms_split_time(N, tf_val = None, t0_val = t0_nu):
    """
    Divide [t0_val, tf_val] into N equal time segments where N >= 2.
    """
    assert N >= 2, "Number of segments N must be >= 2"

    T = float(tf_nu if tf_val is None else tf_val)
    dt = (T - t0_val) / N
    t_nodes = np.linspace(t0_val, T, N + 1)

    return t_nodes, dt
	
def _ms_unpack_z(z, N):
    """
    Variable order:
        z = [Lambda_0, ..., Lambda_{N-1}, X_1, ..., X_{N-1}]
    where Lambda_k and X_k are 7-dimensional.
    lambda0 is fixed to 1.0.
    """
    z = np.asarray(z, dtype=float) 
    n_lam = 7 * N
    n_X   = 7 * (N - 1)
    lam_flat = z[:n_lam]
    X_flat   = z[n_lam:n_lam + n_X]
    l0       = lambda0
    lam_nodes = lam_flat.reshape((N, 7))
    X_nodes   = X_flat.reshape((N - 1, 7))
    return lam_nodes, X_nodes, float(l0)

def _ms_pack_z(lam_nodes, X_nodes):
    """
    Pack [Lambda_0, ..., Lambda_{N-1}, X_1, ..., X_{N-1}] into one vector.
    """
    return np.hstack([lam_nodes.ravel(), X_nodes.ravel()])

# =========================
# Analytical Jacobian
# =========================
# Jacobian Option 1: Calculate the complete STM 14x15
def ms_jacobian(z, N): 
    """
    Return J = ∂res/∂z ∈ R^{(14N-7)×(14N-7)} as a numpy array.

    Variable order:
        z = [Lambda_0, ..., Lambda_{N-1}, X_1, ..., X_{N-1}]
    where each Lambda_k and X_k is 7-dimensional, and lambda0 is fixed.

    Row block order:
        continuity per segment: 14*(N-1)
        terminal conditions: 7
    """
    lam_nodes, X_nodes, l0 = _ms_unpack_z(z, N)
    t_nodes, _ = _ms_split_time(N, tf_nu)

    n_var = 14 * N - 7
    J = np.zeros((n_var, n_var), dtype=float)

    # Column index helpers
    def cidx_lam(k):
        # Lambda_k, k = 0, ..., N-1
        s = 7 * k
        return slice(s, s + 7)

    def cidx_X(k):
        # X_k, k = 1, ..., N-1
        base = 7 * N
        s = base + 7 * (k - 1)
        return slice(s, s + 7)

    row = 0

    # --- Segment 0: from known X0 and variable Lambda_0 to node 1 ---
    X0 = np.array(list(x0_nu) + [m0_nu], dtype=float)
    Xk_14 = np.r_[X0, lam_nodes[0]]

    X_end, L_end, Phi = stm_14x15_ms(
        Xk_14, l0, t_nodes[0], t_nodes[1]
    )
    A = Phi[:, :14]   # ∂[X_end;L_end]/∂[X_start;Lambda_start]
    # b = Phi[:, 14]  # ∂[X_end;L_end]/∂lambda0, not needed since lambda0 is fixed

    # Residual block:
    # [ X_end - X_1 ]
    # [ L_end - Lambda_1 ]
    
    # X0 is fixed, only Lambda_0 contributes from the start of segment 0
    J[row:row+14, cidx_lam(0)] += A[:, 7:14]

    # -I for [X_1, Lambda_1]
    J[row:row+7,      cidx_X(1)]   -= np.eye(7)
    J[row+7:row+14,   cidx_lam(1)] -= np.eye(7)

    row += 14

    # --- Intermediate segments k = 1, ..., N-2 ---
    for k in range(1, N - 1):
        Xk_14 = np.r_[X_nodes[k - 1], lam_nodes[k]]

        X_end, L_end, Phi = stm_14x15_ms(
            Xk_14, l0, t_nodes[k], t_nodes[k + 1]
        )
        A = Phi[:, :14]

        # With respect to (X_k, Lambda_k)
        J[row:row+14, cidx_X(k)]   += A[:, 0:7]
        J[row:row+14, cidx_lam(k)] += A[:, 7:14]

        # With respect to (X_{k+1}, Lambda_{k+1}) -> -I
        J[row:row+7,      cidx_X(k + 1)]   -= np.eye(7)
        J[row+7:row+14,   cidx_lam(k + 1)] -= np.eye(7)

        row += 14

    # --- Terminal conditions from final segment k = N-1 ---
    Xk_14 = np.r_[X_nodes[N - 2], lam_nodes[N - 1]]

    X_end, L_end, Phi = stm_14x15_ms(
        Xk_14, l0, t_nodes[N - 1], t_nodes[N]
    )
    A = Phi[:, :14]

    # Terminal residual in ms_residual():
    # [X_end[0]-xf[0], ..., X_end[5]-xf[5], L_end[6]]
    rows_sel = [0, 1, 2, 3, 4, 5, 13]
    A_sel = A[rows_sel, :]

    # With respect to (X_{N-1}, Lambda_{N-1})
    J[row:row+7, cidx_X(N - 1)]   += A_sel[:, 0:7]
    J[row:row+7, cidx_lam(N - 1)] += A_sel[:, 7:14]

    row += 7

    return J

# ============================================================
#     # Segment 0 now uses stm_14x8() instead of stm_14x15_ms()
#     # Because X0 is fixed, only Lambda_0 is a decision variable here.
#     # stm_14x8 returns sensitivity wrt [Lambda_0, lambda0] -> shape (14, 8)
#     # Since lambda0 is fixed, we only keep the first 7 columns.
#     # ============================================================
#     Phi0 = stm_14x8(
#         lam_nodes[0],
#         eps=epsf,          # use existing global epsilon
#         lambda0=l0,
#         tf=t_nodes[1]
#     )

#     # only keep derivatives wrt Lambda_0, discard d/dlambda0
#     J[row:row+14, cidx_lam(0)] += Phi0[:, 0:7]

#     # -I block for [X_1, Lambda_1]
#     J[row:row+7,      cidx_X(1)]   -= np.eye(7)
#     J[row+7:row+14,   cidx_lam(1)] -= np.eye(7)

#     row += 14

#     # --- Intermediate segments k = 1, ..., N-2 ---
#     for k in range(1, N - 1):
#         Xk_14 = np.r_[X_nodes[k - 1], lam_nodes[k]]

#         X_end, L_end, Phi = stm_14x15_ms(
#             Xk_14, l0, t_nodes[k], t_nodes[k + 1]
#         )
#         A = Phi[:, :14]

#         # With respect to (X_k, Lambda_k)
#         J[row:row+14, cidx_X(k)]   += A[:, 0:7]
#         J[row:row+14, cidx_lam(k)] += A[:, 7:14]

#         # With respect to (X_{k+1}, Lambda_{k+1}) -> -I
#         J[row:row+7,      cidx_X(k + 1)]   -= np.eye(7)
#         J[row+7:row+14,   cidx_lam(k + 1)] -= np.eye(7)

#         row += 14

#     # --- Terminal conditions from final segment k = N-1 ---
#     Xk_14 = np.r_[X_nodes[N - 2], lam_nodes[N - 1]]

#     # Terminal segment STM
#     X_end, L_end, Phi = stm_14x15_ms(
#         Xk_14, l0, t_nodes[N - 1], t_nodes[N]
#     )
#     A = Phi[:, :14]

#     # Terminal residual in ms_residual():
#     # [X_end[0]-xf[0], ..., X_end[5]-xf[5], L_end[6]]
#     rows_sel = [0, 1, 2, 3, 4, 5, 13]
#     A_sel = A[rows_sel, :]

#     # With respect to (X_{N-1}, Lambda_{N-1})
#     J[row:row+7, cidx_X(N - 1)]   += A_sel[:, 0:7]
#     J[row:row+7, cidx_lam(N - 1)] += A_sel[:, 7:14]

#     row += 7

#     return J

# =========================
# Residual function
# =========================
def ms_residual(z, N):
    """
    Return residual of length 14N-7:
      - Continuity: 14 per segment, total 14*(N-1)
      - Terminal: 7 ([x,y,z,vx,vy,vz, lm(tf)])
    """
    lam_nodes, X_nodes, l0 = _ms_unpack_z(z, N)
    t_nodes, _ = _ms_split_time(N, tf_nu)
    X0 = np.array(list(x0_nu) + [m0_nu], dtype=float)

    res = []

    # Segment 0: from known X0 and variable Lambda_0 to node 1
    ta.time = float(t_nodes[0])
    ta.state[:7]   = X0.tolist()
    ta.state[7:14] = np.array(lam_nodes[0], dtype=float).tolist()
    ta.pars[4]     = float(l0)
    ta.propagate_until(float(t_nodes[1]))

    res.extend((np.array(ta.state[:7], dtype=float)   - np.array(X_nodes[0], dtype=float)).tolist())
    res.extend((np.array(ta.state[7:14], dtype=float) - np.array(lam_nodes[1], dtype=float)).tolist())

    # Intermediate segments: k = 1, ..., N-2
    for k in range(1, N - 1):
        ta.time = float(t_nodes[k])
        ta.state[:7]   = np.array(X_nodes[k - 1], dtype=float).tolist()
        ta.state[7:14] = np.array(lam_nodes[k], dtype=float).tolist()
        ta.pars[4]     = float(l0)
        ta.propagate_until(float(t_nodes[k + 1]))

        res.extend((np.array(ta.state[:7], dtype=float)   - np.array(X_nodes[k], dtype=float)).tolist())
        res.extend((np.array(ta.state[7:14], dtype=float) - np.array(lam_nodes[k + 1], dtype=float)).tolist())

    # Final segment: from node N-1 start to tf
    ta.time = float(t_nodes[N - 1])
    ta.state[:7]   = np.array(X_nodes[N - 2], dtype=float).tolist()
    ta.state[7:14] = np.array(lam_nodes[N - 1], dtype=float).tolist()
    ta.pars[4]     = float(l0)
    ta.propagate_until(float(t_nodes[N]))

    X_end = np.array(ta.state[:7], dtype=float)
    L_end = np.array(ta.state[7:14], dtype=float)

    res.extend([
        X_end[0] - xf_nu[0],
        X_end[1] - xf_nu[1],
        X_end[2] - xf_nu[2],
        X_end[3] - xf_nu[3],
        X_end[4] - xf_nu[4],
        X_end[5] - xf_nu[5],
        L_end[6]
    ])

    return np.asarray(res, dtype=float)

# =========================
# Initial guess generation
# =========================
def ms_build_initial_guess(
    N,
    lam0_guess=None,
    rng=None,
    max_attempts=100,
):
    """
    Build an initial guess for forward-forward shooting.
    
    Propagate lam0_guess to initialize the interior nodes.
    When lam0_guess is None, sample up to max_attempts initial guesses.
    """
    assert N >= 2

    X0 = np.array(list(x0_nu) + [m0_nu], dtype=float)
    t_nodes, _ = _ms_split_time(N, tf_nu)

    fixed_lam0 = lam0_guess is not None
    if fixed_lam0:
        lam0_fixed = np.asarray(lam0_guess, dtype=float)
        if lam0_fixed.shape != (7,):
            raise ValueError("lam0_guess must be a 7-dimensional vector.")
        attempts = 1
    else:
        if rng is None:
            rng = np.random
        attempts = int(max_attempts)
        if attempts < 1:
            raise ValueError("max_attempts must be >= 1.")

    for _ in range(attempts):
        if fixed_lam0:
            lam0 = lam0_fixed.copy()
        else:
            lam0 = 0.1 * (2.0 * rng.random(7) - 1.0)
            lam0[-1] = 0.1 * rng.random()

        lam_nodes = np.zeros((N, 7), dtype=float)
        X_nodes = np.zeros((N - 1, 7), dtype=float)

        ta.time = float(t_nodes[0])
        ta.state[:7] = X0.tolist()
        ta.state[7:14] = lam0.tolist()
        ta.pars[4] = float(lambda0)
        lam_nodes[0] = lam0

        valid = True
        for k in range(1, N + 1):
            ta.propagate_until(float(t_nodes[k]))

            state = np.asarray(ta.state, dtype=float)
            if (not np.all(np.isfinite(state))
                    # or state[6] < 0.2
                    # or state[13] < -10.0
                    ):
                valid = False
                break

            if k < N:
                X_nodes[k - 1] = state[:7]
                lam_nodes[k] = state[7:14]

        if valid:
            return _ms_pack_z(lam_nodes, X_nodes)

        if fixed_lam0:
            raise RuntimeError(
                "The supplied lam0_guess cannot generate a valid "
                "forward-forward node guess."
            )

    raise RuntimeError(
        f"Failed to generate a valid multiple-shooting guess "
        f"after {attempts} attempts."
    )

# =========================
# Multiple-shooting solver
# =========================
def solve_multiple_shooting(N, z0=None, use_jac=True, eps=1.0e-5):
    """
    Solve the forward-forward multiple-shooting system.
    """
    if z0 is None:
        try:
            z0 = ms_build_initial_guess(N, lam0_guess=None)
        except RuntimeError:
            return None

    fun = lambda zz: ms_residual(zz, N)

    if use_jac:
        jac = lambda zz: ms_jacobian(zz, N)
        sol = root(
            fun,
            np.asarray(z0, dtype=float),
            jac=jac,
            method='hybr',
            tol = minpack_tol,
            options={
                'xtol': minpack_xtol,
                'maxfev': minpack_maxfev,
                'factor': minpack_factor,
                'diag': [1.]*(7+14*(N-1)),
                'eps': minpack_eps,
            },
        )
    else:
        sol = root(
            fun,
            np.asarray(z0, dtype=float),
            method='hybr',
            tol = minpack_tol,
            options={
                'xtol': minpack_xtol,
                'maxfev': minpack_maxfev,
                'factor': minpack_factor,
                'diag': [1.]*(7+14*(N-1)),
                'eps': minpack_eps,
            },
        )

    return sol


In [11]:
# =========================
# Initial guess generation
# =========================
def fb_mid_build_initial_guess(m0=1.0, lam0_guess=None, rng=None):
    """
    Build z0 = [lam0(7), mf0(1), lam_rf0(3), lam_vf0(3)].
    
    Use lam0_guess for the initial co-states when supplied, and draw
    the terminal co-states from rng.
    """
    if rng is None:
        rng = np.random

    if lam0_guess is None:
        lam0 = 0.1 * (2.0 * rng.random(7) - 1.0)
        lam0[6] = 0.1 * float(rng.random())
    else:
        lam0 = np.asarray(lam0_guess, dtype=float).copy()
        if lam0.shape != (7,):
            raise ValueError("lam0_guess must be a 7-dimensional vector.")

    mf0 = 0.9 * float(m0)
    lrf0 = 0.1 * (2.0 * rng.random(3) - 1.0)
    lvf0 = 0.1 * (2.0 * rng.random(3) - 1.0)

    return np.hstack([lam0, mf0, lrf0, lvf0])

# =========================
# Midpoint forward-backward solver
# =========================
def solve_fb_mid(z0=None, use_jac=True, use_SIMD=False):
    """
    Solve the midpoint forward-backward shooting system.
    """
    if z0 is None:
        z0 = fb_mid_build_initial_guess(m0=m0_nu)

    if not use_SIMD:
        if use_jac:
            r = root(lambda zz: fb_mid_residual(zz),
                    x0 = np.asarray(z0, float),
                    jac=lambda zz: fb_mid_jacobian(zz),
                    method='hybr',
                    # tol=1e-6,
                    # options={
                    #     'xtol':1e-6, 
                    #     'maxfev':50000, 
                    #     'factor':0.1,
                    #     'diag':[1]*7 + [1] + [1]*3 + [1]*3,
                    # }
                    tol = minpack_tol,
                    options = {
                        'xtol': minpack_xtol,
                        'maxfev': minpack_maxfev,
                        'factor': minpack_factor,
                        'diag':[1.]*7 + [1] + [1]*3 + [1]*3,
                        'eps': minpack_eps,
                    },
                )
        else:
            r = root(lambda zz: fb_mid_residual(zz),
                    x0 = np.asarray(z0, float),
                    method='hybr',
                    # tol=1e-6,
                    # options={
                    #     'xtol':1e-6, 
                    #     'maxfev':50000, 
                    #     'factor':0.1,
                    #     'diag':[1]*7 + [1] + [1]*3 + [1]*3,
                    # }
                    
                    tol = minpack_tol,
                    options = {
                        'xtol': minpack_xtol,
                        'maxfev': minpack_maxfev,
                        'factor': minpack_factor,
                        'diag':[1.]*7 + [1] + [1]*3 + [1]*3,
                        'eps': minpack_eps,
                    },
                )
    else: # use SIMD
        if use_jac:
            r = root(lambda zz: fb_mid_residual_SIMD(zz),
                    x0 = np.asarray(z0, float),
                    jac=lambda zz: fb_mid_jacobian_SIMD(zz),
                    method='hybr',
                    # tol=1e-6,
                    # options={
                    #     'xtol':1e-6, 
                    #     'maxfev':50000, 
                    #     'factor':0.1,
                    #     'diag':[1]*7 + [1] + [1]*3 + [1]*3
                    # }
                    tol = minpack_tol,
                    options = {
                        'xtol': minpack_xtol,
                        'maxfev': minpack_maxfev,
                        'factor': minpack_factor,
                        'diag':[1.]*7 + [1] + [1]*3 + [1]*3,
                        'eps': minpack_eps,
                    },
                )
        else:
            r = root(lambda zz: fb_mid_residual_SIMD(zz),
                    x0 = np.asarray(z0, float),
                    method='hybr',
                    # tol=1e-6,
                    # options={
                    #     'xtol':1e-6, 
                    #     'maxfev':50000, 
                    #     'factor':0.1,
                    #     'diag':[1]*7 + [1] + [1]*3 + [1]*3
                    # }
                    
                    tol = minpack_tol,
                    options = {
                        'xtol': minpack_xtol,
                        'maxfev': minpack_maxfev,
                        'factor': minpack_factor,
                        'diag':[1.]*7 + [1] + [1]*3 + [1]*3,
                        'eps': minpack_eps,
                    },
                )
    
    return r


In [12]:
# =========================
# Residual function
# =========================
def fb_mid_residual(z, ratio=0.5):
    """
    Midpoint forward-backward shooting residual, with fixed parameter lambda0 = 1.
    Unknown vector: z = [lam0(7), mf(1), lrf(3), lvf(3)] ∈ R^14
    Residual: g(z) = X_L(t_mid) - X_R(t_mid) ∈ R^14
    """
	
    z    = np.asarray(z, float).reshape(-1)
    lam0 = z[0:7]
    mf   = float(z[7])
    lrf  = z[8:11]
    lvf  = z[11:14]

    t_mid = t0_nu + ratio * (tf_nu-t0_nu)

    # ----- Left arc: t0_nu → t_mid (forward)
    ta.time = t0_nu
    ta.state[:7]   = np.array(list(x0_nu) + [m0_nu], dtype=float)
    ta.state[7:14] = lam0.tolist()
    ta.propagate_until(t_mid)

    XL = np.array(ta.state[:14], dtype=float, copy=True)  # Force copy

    # ----- Right arc: tf_nu → t_mid (backward)
    ta.time = tf_nu
    ta.state[0:6]   = xf_nu
    ta.state[6]     = mf
    ta.state[7:10]  = lrf.tolist()
    ta.state[10:13] = lvf.tolist()
    ta.state[13]    = 0.0  # λ_m,f
    ta.propagate_until(t_mid)
    
    XR  = np.asarray(ta.state[:14], dtype=float)

    g_match = XL - XR

    return g_match

# =========================
# Analytical Jacobian
# =========================
# Jacobian Option 1: Calculate the complete STM 14x15
def fb_mid_jacobian(z, ratio=0.5):
    """
    Analytical Jacobian of the midpoint forward-backward shooting residual
    with fixed l0 = 1.

    ----------
    z : Unknown vector z = [lam0(7), mf(1), lrf(3), lvf(3)]
    ratio : Relative location of the matching point in [0, 1], defined by t_mid = t0_nu + ratio * (tf_nu - t0_nu). Default is 0.5.

    Returns
    -------
    J : ndarray, shape (14, 14)
        Analytical Jacobian of g(z) = X_L(t_mid) - X_R(t_mid)  with respect to z.
    """
	
    z    = np.asarray(z, float).reshape(-1)
    lam0 = z[0:7]
    mf   = float(z[7])
    lrf  = z[8:11]
    lvf  = z[11:14]
    l0   = lambda0

    t_mid = float(t0_nu) + ratio * (float(tf_nu) - float(t0_nu))

    # ---------- Left arc: Φ w.r.t. [X0(14), λ0] ----------
    # Initial 14D state: [x0,y0,z0,vx0,vy0,vz0,m0,  Λ0(7)]
    X0_7 = np.array(list(x0_nu) + [m0_nu], dtype=float)
    X0L   = np.r_[X0_7, lam0]                    # 14D
    
    _, _, PhiL = stm_14x15_ms(X0L, l0, t0_nu, t_mid)
    # Left arc matching sensitive only to Λ0 and λ0:
    dXL_dLam0 = PhiL[:, 7:14]     # (14×7)

    # ---------- Right arc: Φ w.r.t. [X_f(14), λ0] ----------
    Xf = np.zeros(14, dtype=float)
    Xf[0:6]   = xf_nu
    Xf[6]     = mf
    Xf[7:10]  = lrf
    Xf[10:13] = lvf
    Xf[13]    = 0.0                # λ_m,f fixed
    
    _, _, PhiR = stm_14x15_ms(Xf, l0, tf_nu, t_mid)
    # Right arc matching sensitive to m_f, λ_r,f, λ_v,f, λ0:
    dXR_dmf  = PhiR[:, 6:7]        # (14×1)
    dXR_dlrf = PhiR[:, 7:10]       # (14×3)
    dXR_dlvf = PhiR[:, 10:13]      # (14×3)

    # ---------- Assemble J ----------
    J = np.zeros((14, 14), dtype=float)

    # First 14 rows: Matching residual XL - XR
    J[0:14, 0:7]   =  dXL_dLam0
    J[0:14, 7:8]   = -dXR_dmf
    J[0:14, 8:11]  = -dXR_dlrf
    J[0:14, 11:14] = -dXR_dlvf

    return J


In [13]:
# =========================
# Integrator parameter setting (eps and lambda0 are fixed.)
# =========================
# epsf = 1.0e-5
epsf = 1.0e-3
lambda0 = 1.0

pars_b2 = np.asarray([mu_nu, c1, c2, epsf, lambda0], float).reshape(-1, 1).repeat(2, axis=1)
ta_b2.pars[:]              = pars_b2
ta_var_ms_b2.pars[:]       = pars_b2

# ============================================================
# Jacobian function: Unified assembly using stm 14x15
# Column order: z = [ Λ0(7) | m_f(1) | λ_r,f(3) | λ_v,f(3) | λ0(1) ]
# ============================================================
def _extract_phi_b2(state_batch, n_sv=14, n_args=15):
    """
    Extract first-order variational blocks from ta_var_ms_b2.state (shape (n_sv + n_sv*n_args, 2)):
    Return (n_sv, n_args, 2), where the third dimension is the batch dimension (0=left arc, 1=right arc).
    """
    B = state_batch.shape[1]
    assert B == 2
    phi_flat = state_batch[n_sv:n_sv+n_sv*n_args, :]  # (n_sv*n_args, 2)
    out = np.empty((n_sv, n_args, B))
    for b in range(B):
        out[:,:,b] = phi_flat[:, b].reshape( n_sv, n_args)
    return out

# =========================
# Residual function
# =========================
def fb_mid_residual_SIMD(z, ratio=0.5):
    """
    Midpoint forward-backward shooting residual, with fixed parameter lambda0 = 1.
    Single z (shape (14,)): [Λ0(7), mf(1), λr,f(3), λv,f(3)]
    Return g (shape (14,)): [XL(t_mid)-XR(t_mid) (14)]
    """
    z = np.asarray(z, float).reshape(-1)
    lam0 = z[0:7]
    mf   = float(z[7])
    lxf  = z[8:14]
    
    t_mid = t0_nu + ratio * (tf_nu-t0_nu)

    # ---------- Assemble initial values for two columns ----------
    # Left arc column (col 0): t=0, [x0, m0, Λ0]
    x0_col = np.asarray(x0_nu, dtype=float).reshape(6, 1)
    L0  = np.asarray(lam0, float).reshape(7,1)
    XL0 = np.vstack([x0_col, [[m0_nu]], L0]) # (14,1)

    # Right arc column (col 1): t=0 corresponding to tf, set [xf,mf,λxf,λmf=0]
    xf_col = np.asarray(xf_nu, dtype=float).reshape(6, 1)
    Lf  = np.vstack([np.asarray(lxf,float).reshape(6,1),
                     np.zeros((1,1))])          # (7,1) with λm,f=0
    XR0 = np.vstack([xf_col, [[mf]], Lf]) # (14,1)

    X0  = np.concatenate([XL0, XR0], axis=1)    # (14,2)

    # ---------- Reset initial values (official example style) ----------
    ta_b2.set_time([t0_nu, tf_nu])
    ta_b2.state[:] = X0

    # ---------- Target time: left column t_mid, right column t_mid - tf (negative time) ----------
    ta_b2.propagate_until([t_mid, t_mid])

    XL = ta_b2.state[:14, 0].copy()   # Left arc at t_mid
    XR = ta_b2.state[:14, 1].copy()   # Right arc at t_mid

    g_match = XL - XR
    
    return g_match

def fb_mid_jacobian_SIMD(z, ratio=0.5):
    """
    Analytical Jacobian of the midpoint forward-backward shooting residual
    with fixed l0 = 1.

    ----------
    z : Unknown vector z = [lam0(7), mf(1), lrf(3), lvf(3)]
    ratio : Relative location of the matching point in [0, 1], defined by t_mid = t0_nu + ratio * (tf_nu - t0_nu). Default is 0.5.

    Returns
    -------
    J : ndarray, shape (14, 14)
        Analytical Jacobian of g(z) = X_L(t_mid) - X_R(t_mid)  with respect to z.
    """
	
    z = np.asarray(z, float).reshape(-1)
    lam0 = z[0:7]
    mf   = float(z[7])
    lxf  = z[8:14]

    t_mid = float(t0_nu) + ratio * (float(tf_nu) - float(t0_nu))

    # ---------- Assemble base states for two columns ----------
    # Left arc column (col 0): t=0, [x0, m0, Λ0]
    x0_col = np.asarray(x0_nu, dtype=float).reshape(6, 1)
    L0  = np.asarray(lam0, float).reshape(7,1)
    XL0 = np.vstack([x0_col, [[m0_nu]], L0]) # (14,1)

    # Right arc column (col 1): t=0 corresponding to tf, set [xf,mf,λxf,λmf=0]
    xf_col = np.asarray(xf_nu, dtype=float).reshape(6, 1)
    Lf  = np.vstack([np.asarray(lxf,float).reshape(6,1),
                     np.zeros((1,1))])          # (7,1) with λm,f=0
    XR0 = np.vstack([xf_col, [[mf]], Lf]) # (14,1)

    X0  = np.concatenate([XL0, XR0], axis=1)    # (14,2)

    # ---------- Reset initial values (official example style) ----------
    ta_var_ms_b2.set_time([t0_nu, tf_nu])
    ta_var_ms_b2.state[:14, :] = X0
    # Construct column vector and explicitly tile into two columns, ensuring assignment shape is (14*15, 2)
    Phi0 = np.hstack([np.eye(14), np.zeros((14, 1))]).ravel()
    Phi0_col = np.asarray(Phi0).reshape(-1, 1).copy()                       # (14*15, 1)
    ta_var_ms_b2.state[14:14+14*15, :] = np.hstack([Phi0_col, Phi0_col])  # Same initial Φ for both columns
    ta_var_ms_b2.propagate_until([t_mid, t_mid])

    # ---------- Extract Φ (two columns) ----------
    phi = _extract_phi_b2(ta_var_ms_b2.state, n_sv=14, n_args=15)
    # Indices:   0..5:x..vz, 6:m, 7..13:lx..lm, 14:par[4]=λ0

    # ---------- Assemble Jacobian ----------
    J = np.zeros((14, 14))

    # First 14 rows: Matching items
    # d(XL - XR)/d[Λ0(7)]
    J[:, 0:7]   =  phi[:, 7:14, 0]     # Left column's 7 columns for λ
    # d(XL - XR)/d mf
    J[:, 7]     = -phi[:, 6,    1]     # Right column's column for m
    # d(XL - XR)/d λr,f(3)
    J[:, 8:11]  = -phi[:, 7:10, 1]     # Right column's lx..lz
    # d(XL - XR)/d λv,f(3)
    J[:, 11:14] = -phi[:, 10:13,1]     # Right column's lvx..lvz

    return J


## Method 1: single shooting


In [14]:
# =========================
# Initialize solving time and best solution
# =========================
# epsf = 1.0e-5
epsf = 1.0e-3
lambda0 = 1.0

count_solved = 0
total_time = 0.0
total_nfev = 0
converge_nfev = 0

global_lambda = None
global_mf = -1.0

# =========================
# Random initial guess loop
# =========================


In [15]:
records = []
for i in range(repeat_num_minpack):
    accepted_before = count_solved
    nfev_before = total_nfev
    r = None
    # Common initial co-state for trial i
    x0 = lam0_pool[i].copy()

    time_start = time.perf_counter()

    # Use root's args parameter to pass fixed arguments
    r = root(
        shooting,
        x0,
        args = (epsf, lambda0, False, None),   # order: lam, eps, lambda0, out, Time_out
        method = 'hybr',
        tol = minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        }
    )

    time_end = time.perf_counter()
    total_time += time_end - time_start
    total_nfev += r['nfev']

    # Success criterion: solver converged AND residual below threshold
    res = np.linalg.norm(r['fun'])
    if r['success'] and res < minpack_error:
        count_solved += 1
        converge_nfev += r['nfev']

        print(f"counter: {count_solved}/{i} nfev: {r['nfev']}", end='', flush=True)
        # Call one more time with out = True to output terminal mass
        shooting(r['x'], epsf, lambda0, out=True, Time_out=None)
        # Save the best solution
        final_mass = ta.state[6]*MUnit
        if final_mass>global_mf:
            global_mf = final_mass
            global_lambda = r['x']
    save_trial(i, r, count_solved > accepted_before, total_nfev - nfev_before)

# =========================
# Output convergence rate and average convergence time
# =========================
if count_solved < 1:
    print(f"Fail to solve the problem!")
else:
    print(f"Succeed to solve the problem!")
    print(f"Solved: {count_solved} in {repeat_num_minpack} ({count_solved/repeat_num_minpack:.4f}), terminal mass: {global_mf:.4f}kg")
    print(f"Average number of function calls for Minpack: {total_nfev/repeat_num_minpack:.0f}")
    print(f"Average number of function calls for convergent solution: {converge_nfev/count_solved:.0f}")
    print("global_lambda:", global_lambda)

records_by_method['Single_FD'] = records


counter: 1/37 nfev: 335

  Final Mass: 1035.657052701444


counter: 2/67 nfev: 381

  Final Mass: 1258.9588855337086


counter: 3/133 nfev: 230

  Final Mass: 1289.9020037982627


... (242 outputs left out)

  Final Mass: 1258.9588855337647


counter: 136/4986 nfev: 546

  Final Mass: 1258.9588855332604


Succeed to solve the problem!
Solved: 136 in 5000 (0.0272), terminal mass: 1289.9020kg
Average number of function calls for Minpack: 142
Average number of function calls for convergent solution: 304
global_lambda: [ 0.5980619823523473  0.1525985906790719  0.2581872881464975
 -0.1508729531201627  0.5962339628995333 -0.1321461044862927
  0.141409721660566 ]


In [16]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['Single_FD']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()


In [17]:
%%timeit -n 1 -r 7 -v timing_all set_timing_priority()
for i in timing_ids:
    sample_cpu()
    # Common initial co-state for trial i
    x0 = lam0_pool[i].copy()

    # Use root's args parameter to pass fixed arguments
    r = root(
        shooting,
        x0,
        args = (epsf, lambda0, False, None),   # order: lam, eps, lambda0, out, Time_out
        method = 'hybr',
        tol = minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        }
    )


6.75 s ± 24.7 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [18]:
%%timeit -n 1 -r 7 -v timing_successful set_timing_priority()
for i in successful_ids:
    sample_cpu()
    # Common initial co-state for trial i
    x0 = lam0_pool[i].copy()

    # Use root's args parameter to pass fixed arguments
    r = root(
        shooting,
        x0,
        args = (epsf, lambda0, False, None),   # order: lam, eps, lambda0, out, Time_out
        method = 'hybr',
        tol = minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        }
    )


215 ms ± 2.02 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [19]:
%%timeit -n 1 -r 7 -v timing_failed set_timing_priority()
for i in failed_ids:
    sample_cpu()
    # Common initial co-state for trial i
    x0 = lam0_pool[i].copy()

    # Use root's args parameter to pass fixed arguments
    r = root(
        shooting,
        x0,
        args = (epsf, lambda0, False, None),   # order: lam, eps, lambda0, out, Time_out
        method = 'hybr',
        tol = minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        }
    )


6.54 s ± 24.7 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [20]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['Single_FD'] = dict(
    method='Single shooting, FD', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['Single_FD'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['Single_FD']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)


## Method 2: single shooting + gradient


In [14]:
# =========================
# Initialize solving time and best solution
# =========================
# epsf = 1.0e-5
epsf = 1.0e-3
lambda0 = 1.0

count_solved = 0
total_time = 0.0
total_nfev = 0
converge_nfev = 0

global_lambda = None
global_mf = -1.0

# =========================
# Random initial guess loop
# =========================


In [15]:
records = []
for i in range(repeat_num_minpack):
    accepted_before = count_solved
    nfev_before = total_nfev
    r = None
    # Common initial co-state for trial i
    x0 = lam0_pool[i].copy()

    time_start = time.perf_counter()

    r = root(
        lambda lam: shooting(lam, epsf, lambda0, False, None),
        x0 = x0,
        jac = lambda lam: shooting_jac(lam, epsf, lambda0),
        method = 'hybr',
        tol = minpack_tol,
        options = {
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        },
    )

    time_end = time.perf_counter()
    total_time += time_end - time_start
    total_nfev += r['nfev']

    res = np.linalg.norm(r['fun'])

    if r['success'] and res < minpack_error:
        count_solved += 1
        converge_nfev += r['nfev']

        print(f"counter: {count_solved}/{i} nfev: {r['nfev']}", end='', flush=True)
        # Call one more time with out = True to output terminal mass
        shooting(r['x'], epsf, lambda0, out = True, Time_out = None)
        # Save the best solution
        final_mass = ta.state[6] * MUnit
        if final_mass > global_mf:
            global_mf = final_mass
            global_lambda = r['x']
    save_trial(i, r, count_solved > accepted_before, total_nfev - nfev_before)

# =========================
# Output convergence rate and average convergence time
# =========================
if count_solved < 1:
    print(f"Fail to solve the problem!")
else:
    print(f"Succeed to solve the problem!")
    print(f"Solved: {count_solved} in {repeat_num_minpack} ({count_solved/repeat_num_minpack:.4f}), terminal mass: {global_mf:.4f}kg")
    print(f"Average number of function calls for Minpack: {total_nfev/repeat_num_minpack:.0f}")
    print(f"Average number of function calls for convergent solution: {converge_nfev/count_solved:.0f}")
    print("global_lambda:", global_lambda)
records_by_method['Single_AJ'] = records


counter: 1/36 nfev: 93

  Final Mass: 1258.9588855338607
counter: 2/37 nfev: 125

  Final Mass: 1035.6570527030503


counter: 3/68 nfev: 159

  Final Mass: 1289.902003795388
counter: 4/72 nfev: 136

  Final Mass: 1289.902003798765


... (313 outputs left out)

  Final Mass: 1258.958885526244


counter: 165/4992 nfev: 129

  Final Mass: 1258.9588855338957


Succeed to solve the problem!
Solved: 165 in 5000 (0.0330), terminal mass: 1289.9020kg
Average number of function calls for Minpack: 66
Average number of function calls for convergent solution: 154
global_lambda: [ 0.598061982340796   0.1525985906706792  0.2581872881292199
 -0.150872953122554   0.5962339628854441 -0.132146104459943
  0.1414097216794125]


In [16]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['Single_AJ']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()


In [17]:
%%timeit -n 1 -r 7 -v timing_all set_timing_priority()
for i in timing_ids:
    sample_cpu()
    # Common initial co-state for trial i
    x0 = lam0_pool[i].copy()

    r = root(
        lambda lam: shooting(lam, epsf, lambda0, False, None),
        x0 = x0,
        jac = lambda lam: shooting_jac(lam, epsf, lambda0),
        method = 'hybr',
        tol = minpack_tol,
        options = {
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        },
    )


9.12 s ± 151 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [18]:
%%timeit -n 1 -r 7 -v timing_successful set_timing_priority()
for i in successful_ids:
    sample_cpu()
    # Common initial co-state for trial i
    x0 = lam0_pool[i].copy()

    r = root(
        lambda lam: shooting(lam, epsf, lambda0, False, None),
        x0 = x0,
        jac = lambda lam: shooting_jac(lam, epsf, lambda0),
        method = 'hybr',
        tol = minpack_tol,
        options = {
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        },
    )


527 ms ± 2.19 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [19]:
%%timeit -n 1 -r 7 -v timing_failed set_timing_priority()
for i in failed_ids:
    sample_cpu()
    # Common initial co-state for trial i
    x0 = lam0_pool[i].copy()

    r = root(
        lambda lam: shooting(lam, epsf, lambda0, False, None),
        x0 = x0,
        jac = lambda lam: shooting_jac(lam, epsf, lambda0),
        method = 'hybr',
        tol = minpack_tol,
        options = {
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        },
    )


8.55 s ± 108 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [20]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['Single_AJ'] = dict(
    method='Single shooting, AJ', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['Single_AJ'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['Single_AJ']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)


## Method 3: forward-forward shooting + gradient


In [14]:
# =========================
# Forward-forward shooting
# =========================
use_Jacobian = True
N = 2

# =========================
# Initialize solving time and best solution
# =========================
# epsf = 1.0e-5
# lambda0 = 1.0

count_solved = 0
total_time = 0.0
total_nfev = 0
converge_nfev = 0
build_failed = 0
solver_runs = 0

global_z = None
global_lambda = None
global_mf = -1.0

# =========================
# Random initial guess loop
# =========================


In [15]:
records = []
for i in range(repeat_num_minpack):
    accepted_before = count_solved
    nfev_before = total_nfev
    r = None
    # Build the node initial guess from lam0_pool[i].
    
    try:
        z0_ms = ms_build_initial_guess(
            N,
            lam0_guess=lam0_pool[i],
        )
    except RuntimeError:
        build_failed += 1
        save_trial(i, r, count_solved > accepted_before, total_nfev - nfev_before)
        continue

    solver_runs += 1
    time_start = time.perf_counter()

    r = solve_multiple_shooting(
        N,
        z0=z0_ms,
        use_jac=use_Jacobian,
        eps=epsf,
    )

    time_end = time.perf_counter()
    total_time += time_end - time_start

    if r is None:
        save_trial(i, r, count_solved > accepted_before, total_nfev - nfev_before)
        continue

    total_nfev += r['nfev']

    res = np.linalg.norm(r['fun'])

    if r['success'] and res < minpack_error:
        count_solved += 1
        converge_nfev += r['nfev']

        print(f"counter: {count_solved}/{i} nfev: {r['nfev']}", end='', flush=True)

        lam_nodes, X_nodes, l0 = _ms_unpack_z(r['x'], N)

        # Output final mass
        shooting(lam_nodes[0], epsf, lambda0, out = True, Time_out = None)

        # Save the best solution
        final_mass = ta.state[6] * MUnit
        if final_mass > global_mf:
            global_mf = final_mass
            global_z = r['x'].copy()
            global_lambda = lam_nodes[0].copy()
    save_trial(i, r, count_solved > accepted_before, total_nfev - nfev_before)

# =========================
# Output convergence rate and average convergence time
# =========================
if count_solved < 1:
    print(f"Fail to solve the problem!")
else:
    print(f"Succeed to solve the problem!")
    print(f"Solved: {count_solved} in {repeat_num_minpack} ({count_solved / repeat_num_minpack:.4f}), terminal mass: {global_mf:.4f}kg")
    print(f"Initial-node construction failures: {build_failed}")
    print(f"Solver runs after node construction: {solver_runs}")
    if solver_runs > 0:
        print(f"Average number of function calls per solver run: {total_nfev / solver_runs:.0f}")
    print(f"Average number of function calls for convergent solution: {converge_nfev / count_solved:.0f}")
    print("global_lambda:", global_lambda)
    print("global_z:", global_z)
records_by_method['FF_AJ'] = records


counter: 1/206 nfev: 271

  Final Mass: 1035.657052701209


counter: 2/401 nfev: 160

  Final Mass: 1035.6570527014471


counter: 3/1238 nfev: 237

  Final Mass: 1035.6570527014303


... (23 outputs left out)

  Final Mass: 1035.6570527014437


counter: 16/4773 nfev: 414

  Final Mass: 1035.6570527015392


Succeed to solve the problem!
Solved: 16 in 5000 (0.0032), terminal mass: 1035.6571kg
Initial-node construction failures: 0
Solver runs after node construction: 5000
Average number of function calls per solver run: 69
Average number of function calls for convergent solution: 262
global_lambda: [-0.6875275933119586 -0.2433285015316995  0.1257975797252424
  0.0737804177691551 -0.7061338510278579 -0.0484493543814179
  0.3745789068709183]
global_z: [-0.6875275933119586 -0.2433285015316995  0.1257975797252424
  0.0737804177691551 -0.7061338510278579 -0.0484493543814179
  0.3745789068709183  0.0380241561867579 -0.0981968828354002
  0.0901764227185695 -0.2142161745531937  0.1018506058847815
  0.0069797711025481  0.2916323023125985  1.4607842401307451
 -0.2078314835833492 -0.0316061090741106  0.0176645727554736
  0.7891753596583777  0.006519144751549   0.8925349967378666]


In [16]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(50)
accepted = np.array([r['accepted'] for r in records_by_method['FF_AJ']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()


In [17]:
%%timeit -n 1 -r 7 -v timing_all set_timing_priority()
for i in timing_ids:
    sample_cpu()
    # Build the node initial guess from lam0_pool[i].
    
    try:
        z0_ms = ms_build_initial_guess(
            N,
            lam0_guess=lam0_pool[i],
        )
    except RuntimeError:
        continue

    r = solve_multiple_shooting(
        N,
        z0=z0_ms,
        use_jac=use_Jacobian,
        eps=epsf,
    )


1.37 s ± 2.51 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [18]:
%%timeit -n 1 -r 7 -v timing_successful set_timing_priority()
for i in successful_ids:
    sample_cpu()
    # Build the node initial guess from lam0_pool[i].
    
    try:
        z0_ms = ms_build_initial_guess(
            N,
            lam0_guess=lam0_pool[i],
        )
    except RuntimeError:
        continue

    r = solve_multiple_shooting(
        N,
        z0=z0_ms,
        use_jac=use_Jacobian,
        eps=epsf,
    )


The slowest run took 5.38 times longer than the fastest. This could mean that an intermediate result is being cached.
845 ns ± 552 ns per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [19]:
%%timeit -n 1 -r 7 -v timing_failed set_timing_priority()
for i in failed_ids:
    sample_cpu()
    # Build the node initial guess from lam0_pool[i].
    
    try:
        z0_ms = ms_build_initial_guess(
            N,
            lam0_guess=lam0_pool[i],
        )
    except RuntimeError:
        continue

    r = solve_multiple_shooting(
        N,
        z0=z0_ms,
        use_jac=use_Jacobian,
        eps=epsf,
    )


1.37 s ± 3.37 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [20]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['FF_AJ'] = dict(
    method='Forward-forward, AJ', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['FF_AJ'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['FF_AJ']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)


## Method 4: forward-backward shooting + gradient


In [14]:
# =========================
# Forward-backward shooting
# =========================
use_Jacobian = True

# =========================
# Initialize solving time and best solution
# =========================
count_solved = 0
total_time = 0.0
total_nfev = 0
converge_nfev = 0

global_z = None
global_lambda = None
global_mf = -1.0

# =========================
# Random initial guess loop
# =========================


In [15]:
records = []
for i in range(repeat_num_minpack):
    accepted_before = count_solved
    nfev_before = total_nfev
    r = None
    z0_fb = fb_z0_pool[i].copy()

    time_start = time.perf_counter()

    r = solve_fb_mid(
        z0=z0_fb,
        use_jac=use_Jacobian,
        use_SIMD=False,
    )

    time_end = time.perf_counter()
    total_time += time_end - time_start
    total_nfev += r.nfev

    res = np.linalg.norm(r.fun)
    if r.success and res < minpack_error:
        count_solved += 1
        converge_nfev += r.nfev
        print(f"counter: {count_solved}/{i} nfev: {r.nfev}", end=' ', flush=True)

        lam0_sol = r.x[:7].copy()

        # Output final mass
        shooting(lam0_sol, eps=epsf, lambda0=lambda0, out=True)

        # Save the best solution
        final_mass = ta.state[6] * MUnit
        if final_mass > global_mf:
            global_mf = final_mass
            global_z = r.x.copy()
            global_lambda = lam0_sol.copy()
    save_trial(i, r, count_solved > accepted_before, total_nfev - nfev_before)

# =========================
# Output convergence rate and average convergence time
# =========================
if count_solved < 1:
    print("Fail to solve the problem!")
else:
    print("Succeed to solve the problem!")
    print(f"Solved: {count_solved} in {repeat_num_minpack} ({count_solved / repeat_num_minpack:.4f}), terminal mass: {global_mf:.4f}kg")
    print(f"Average number of function calls for Minpack: {total_nfev / repeat_num_minpack:.0f}")
    print(f"Average number of function calls for convergent solution: {converge_nfev / count_solved:.0f}")
    print("global_lambda:", global_lambda)
    print("global_z:", global_z)
records_by_method['FB_AJ'] = records


counter: 1/3 nfev: 150 

  Final Mass: 1289.9020038057154


counter: 2/17 nfev: 220 

  Final Mass: 1289.9020038045426


counter: 3/21 nfev: 116 

  Final Mass: 1258.9588855340094
counter: 4/22 nfev: 198 

... (1,661 outputs left out)

counter: 889/4981 nfev: 88 

  Final Mass: 1289.9020038078818
counter: 890/4983 nfev: 129 

  Final Mass: 1289.9020037350417


Succeed to solve the problem!
Solved: 890 in 5000 (0.1780), terminal mass: 1289.9020kg
Average number of function calls for Minpack: 88
Average number of function calls for convergent solution: 171
global_lambda: [ 0.5980619825801166  0.1525985907331479  0.2581872880565004
 -0.1508729531387768  0.5962339630429817 -0.1321461042735103
  0.1414097217635449]
global_z: [ 0.5980619825801166  0.1525985907331479  0.2581872880565004
 -0.1508729531387768  0.5962339630429817 -0.1321461042735103
  0.1414097217635449  0.8599346691244972 -0.5171177935016354
  0.9736266554776662 -0.2211019808929788 -0.6062449361427742
 -0.3032202636267811 -0.1573811416610204]


In [16]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['FB_AJ']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()


In [17]:
%%timeit -n 1 -r 7 -v timing_all set_timing_priority()
for i in timing_ids:
    sample_cpu()
    z0_fb = fb_z0_pool[i].copy()

    r = solve_fb_mid(
        z0=z0_fb,
        use_jac=use_Jacobian,
        use_SIMD=False,
    )


21.8 s ± 77.3 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [18]:
%%timeit -n 1 -r 7 -v timing_successful set_timing_priority()
for i in successful_ids:
    sample_cpu()
    z0_fb = fb_z0_pool[i].copy()

    r = solve_fb_mid(
        z0=z0_fb,
        use_jac=use_Jacobian,
        use_SIMD=False,
    )


7.66 s ± 5.16 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [19]:
%%timeit -n 1 -r 7 -v timing_failed set_timing_priority()
for i in failed_ids:
    sample_cpu()
    z0_fb = fb_z0_pool[i].copy()

    r = solve_fb_mid(
        z0=z0_fb,
        use_jac=use_Jacobian,
        use_SIMD=False,
    )


14.3 s ± 185 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [20]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['FB_AJ'] = dict(
    method='Forward-backward, AJ', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['FB_AJ'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['FB_AJ']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)


## Method 5: forward-backward shooting with SIMD + gradient


In [14]:
# =========================
# Forward-backward shooting
# =========================
use_Jacobian = True

# =========================
# Initialize solving time and best solution
# =========================
count_solved = 0
total_time = 0.0
total_nfev = 0
converge_nfev = 0

global_z = None
global_lambda = None
global_mf = -1.0

# =========================
# Random initial guess loop
# =========================


In [15]:
records = []
for i in range(repeat_num_minpack):
    accepted_before = count_solved
    nfev_before = total_nfev
    r = None
    # Initial guess from fb_z0_pool[i].
    z0_fb = fb_z0_pool[i].copy()

    time_start = time.perf_counter()

    r = solve_fb_mid(
        z0=z0_fb,
        use_jac=use_Jacobian,
        use_SIMD=True,
    )

    time_end = time.perf_counter()
    total_time += time_end - time_start
    total_nfev += r.nfev

    res = np.linalg.norm(r.fun)
    if r.success and res < minpack_error:
        count_solved += 1
        converge_nfev += r.nfev
        print(f"counter: {count_solved}/{i} nfev: {r.nfev}", end=' ', flush=True)

        lam0_sol = r.x[:7].copy()

        # Output final mass
        shooting(lam0_sol, eps=epsf, lambda0=lambda0, out=True)

        # Save the best solution
        final_mass = ta.state[6] * MUnit
        if final_mass > global_mf:
            global_mf = final_mass
            global_z = r.x.copy()
            global_lambda = lam0_sol.copy()
    save_trial(i, r, count_solved > accepted_before, total_nfev - nfev_before)

# =========================
# Output convergence rate and average convergence time
# =========================
if count_solved < 1:
    print("Fail to solve the problem!")
else:
    print("Succeed to solve the problem!")
    print(f"Solved: {count_solved} in {repeat_num_minpack} ({count_solved / repeat_num_minpack:.4f}), terminal mass: {global_mf:.4f}kg")
    print(f"Average number of function calls for Minpack: {total_nfev / repeat_num_minpack:.0f}")
    print(f"Average number of function calls for convergent solution: {converge_nfev / count_solved:.0f}")
    print("global_lambda:", global_lambda)
    print("global_z:", global_z)
records_by_method['FB_SIMD_AJ'] = records


counter: 1/3 nfev: 150 

  Final Mass: 1289.9020038049307


counter: 2/17 nfev: 175 

  Final Mass: 1289.9020038001672
counter: 3/21 nfev: 116 

  Final Mass: 1258.9588855340578


counter: 4/35 nfev: 135 

... (1,591 outputs left out)

  Final Mass: 1289.9020038047654


counter: 892/4983 nfev: 135 

  Final Mass: 1289.9020038042015


Succeed to solve the problem!
Solved: 892 in 5000 (0.1784), terminal mass: 1289.9020kg
Average number of function calls for Minpack: 88
Average number of function calls for convergent solution: 172
global_lambda: [ 0.5980619823676641  0.1525985906745151  0.2581872881118437
 -0.1508729531262956  0.5962339628985931 -0.1321461044326554
  0.141409721693253 ]
global_z: [ 0.5980619823676641  0.1525985906745151  0.2581872881118437
 -0.1508729531262956  0.5962339628985931 -0.1321461044326554
  0.141409721693253   0.8599346691869453 -0.5171177933889857
  0.9736266552848653 -0.2211019811043905 -0.6062449359656473
 -0.3032202635407285 -0.1573811416753552]


In [16]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['FB_SIMD_AJ']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()


In [17]:
%%timeit -n 1 -r 7 -v timing_all set_timing_priority()
for i in timing_ids:
    sample_cpu()
    # Initial guess from fb_z0_pool[i].
    z0_fb = fb_z0_pool[i].copy()

    r = solve_fb_mid(
        z0=z0_fb,
        use_jac=use_Jacobian,
        use_SIMD=True,
    )


14.7 s ± 265 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [18]:
%%timeit -n 1 -r 7 -v timing_successful set_timing_priority()
for i in successful_ids:
    sample_cpu()
    # Initial guess from fb_z0_pool[i].
    z0_fb = fb_z0_pool[i].copy()

    r = solve_fb_mid(
        z0=z0_fb,
        use_jac=use_Jacobian,
        use_SIMD=True,
    )


5.18 s ± 26.9 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [19]:
%%timeit -n 1 -r 7 -v timing_failed set_timing_priority()
for i in failed_ids:
    sample_cpu()
    # Initial guess from fb_z0_pool[i].
    z0_fb = fb_z0_pool[i].copy()

    r = solve_fb_mid(
        z0=z0_fb,
        use_jac=use_Jacobian,
        use_SIMD=True,
    )


9.52 s ± 146 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [20]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['FB_SIMD_AJ'] = dict(
    method='Forward-backward SIMD, AJ', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['FB_SIMD_AJ'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['FB_SIMD_AJ']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)


## Results


In [4]:
headings = ['Method', 'Expected first solution [ms]', 'Successful mean [ms]',
            'Failed mean [ms]', 'All-trial mean [ms]', 'Converged', 'NFEV, successful']
table = ['| ' + ' | '.join(headings) + ' |', '| ' + ' | '.join(['---'] * len(headings)) + ' |']
for row in results_by_method.values():
    values = [row['method']]
    values += [f"{row[key]:.3f}" if np.isfinite(row[key]) else '—'
               for key in ('first_ms', 'successful_ms', 'failed_ms', 'all_ms')]
    values += [f"{row['accepted']}/{row['trials']}", f"{row['nfev_successful']:.1f}"]
    table.append('| ' + ' | '.join(values) + ' |')
display(Markdown('\n'.join(table)))


| Method | Expected first solution [ms] | Successful mean [ms] | Failed mean [ms] | All-trial mean [ms] | Converged | NFEV, successful |
| --- | --- | --- | --- | --- | --- | --- |
| Single shooting, FD | 496.504 | 30.672 | 13.256 | 13.505 | 136/5000 | 303.6 |
| Single shooting, AJ | 552.552 | 30.980 | 17.704 | 18.234 | 165/5000 | 153.8 |
| Forward-forward, AJ | 8547.233 | — | 27.324 | 27.351 | 16/5000 | 262.2 |
| Forward-backward, AJ | 244.937 | 77.415 | 35.694 | 43.599 | 890/5000 | 170.7 |
| Forward-backward SIMD, AJ | 165.201 | 53.416 | 23.612 | 29.472 | 892/5000 | 171.9 |